# 12. Constraint별 penalty coefficient 균형 실험

## 문제 인식

본 실험(notebook 04, 05)은 모든 constraint에 **동일한 lambda**를 사용했습니다. 그런데 constraint마다 계수의 크기가 다르므로, 같은 lambda를 써도 하드웨어가 보는 제약 강도는 크게 달라집니다.

| constraint | 식 | 제곱 후 계수 규모 |
|---|---|---|
| capacity | `sum_i d_i x_ij - s_j y_j + slack = 0` | `lambda * s_j^2` |
| assignment | `sum_j x_ij - 1 = 0` | `lambda` |

두 penalty의 비는 대략 `2 / s_max^2`입니다. 8x8에서는 약 `10^-4`, 즉 **assignment가 capacity보다 1만 배 약합니다.**

auto_scale 이후로 보면 assignment 계수가 `4.3e-4`인데 ICE 노이즈는 `10^-2` 수준입니다. **QPU는 capacity만 보고 assignment는 사실상 보지 못합니다.** SS의 feasible 비율이 0.2%였던 이유가 여기 있을 수 있습니다.

## 이 실험이 확인하는 것

capacity penalty를 고정한 채 assignment penalty에만 배수를 곱해가며 무엇이 달라지는지 봅니다.

## 원칙 위반이 아닌 이유

penalty method가 요구하는 것은 각 constraint에 대해 `lambda_k > Z_ub`입니다. **모든 constraint가 같은 lambda를 써야 한다는 요구는 없습니다.** assignment penalty를 키우는 것은 그 조건을 더 강하게 만들 뿐이므로 ground state가 feasible이라는 보장은 유지됩니다.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("설정 로드 완료")


설정 로드 완료


## 실험 설정

아래 네 값만 바꾸면 다른 조건으로 다시 돌릴 수 있습니다.

- `QA_DRYRUN` — **True면 오프라인, False면 실제 QPU**
  - 오프라인은 Pegasus P16 그래프에 minorminer를 직접 실행해 embedding을 확인하고, sampling은 SA로 대체합니다. **노이즈를 주지 않으므로 이상적 조건의 상한**을 봅니다.
  - 두 모드의 차이가 곧 하드웨어가 잃는 양입니다.
- `CAPACITY_MULTIPLIER` — capacity penalty에 쓸 배수. `U_obj`에 곱해지며 기존 `config['penalty']['margin']`과 같은 의미입니다. **1.1이면 기존 실험이 그대로 재현됩니다.**
- `RATIOS` — assignment penalty가 capacity penalty의 몇 배인지. `1.0`이 기존 설정입니다.
- `TARGETS` — 실험할 (instance, formulation) 목록

In [1]:
QA_DRYRUN = False          # True: 오프라인(노이즈 없음) / False: 실제 QPU
CAPACITY_MULTIPLIER = 1.1  # U_obj에 곱하는 배수. 1.1 = 기존 실험 설정
RATIOS = (1.0, 10.0, 50.0, 100.0, 200.0, 500.0, 1000.0, 2000.0, 5000.0)
TARGETS = [("4x4", "SS"), ("6x6", "SS"), ("8x8", "SS"), ("15x15", "SS")]

print(f"모드              : {'오프라인 (Pegasus + SA, 노이즈 없음)' if QA_DRYRUN else '실제 QPU'}")
print(f"capacity 배수     : {CAPACITY_MULTIPLIER}")
print(f"assignment 비율   : {RATIOS}")
print(f"대상              : {TARGETS}")

모드              : 실제 QPU
capacity 배수     : 1.1
assignment 비율   : (1.0, 10.0, 50.0, 100.0, 200.0, 500.0, 1000.0, 2000.0, 5000.0)
대상              : [('4x4', 'SS'), ('6x6', 'SS'), ('8x8', 'SS'), ('15x15', 'SS')]


## 공짜로 올릴 수 있는 상한

assignment penalty를 키워도 **일정 지점까지는 coefficient range가 전혀 변하지 않습니다.** capacity의 최대 계수가 `lambda * s_max^2`이고 assignment의 최대 계수가 `2 * lambda * ratio`이므로, `ratio < s_max^2 / 2`인 동안은 여전히 capacity가 최대이기 때문입니다.

그 지점을 넘으면 assignment가 최대가 되어 range가 커지기 시작합니다. 즉 아래 값이 **비용 없이 올릴 수 있는 상한**입니다.

In [4]:
from src.data_generator import CFLPInstance
from src.cflp_ss import objective_upper_bound
from src import penalty_balance as PB

gurobi = pd.read_csv(RAW_DIR / "gurobi_results.csv")
rows = []
for name, formulation in TARGETS:
    instance = CFLPInstance.load(DATA_DIR / f"{name}.json")
    model_name = "SS" if formulation == "SS" else "MS"
    reference = float(
        gurobi[(gurobi["instance"] == name)
               & (gurobi["gurobi_model"] == model_name)]["objective"].iloc[0]
    )
    upper = objective_upper_bound(instance)
    rows.append({
        "instance": name,
        "formulation": formulation,
        "Z_ub": round(reference, 1),
        "U_obj": round(upper, 1),
        "lambda_capacity": round(CAPACITY_MULTIPLIER * upper, 1),
        "lambda/Z_ub": round(CAPACITY_MULTIPLIER * upper / reference, 2),
        "free_ratio_limit": round(PB.balanced_ratio(instance), 0),
    })
pd.DataFrame(rows)

,instance,formulation,Z_ub,U_obj,lambda_capacity,lambda/Z_ub,free_ratio_limit
0,4x4,SS,2416.8,5379.2,5917.2,2.45,2450.0
1,6x6,SS,4886.8,11791.6,12970.8,2.65,2380.0
2,8x8,SS,3952.5,16386.0,18024.6,4.56,10658.0
3,15x15,SS,8489.2,56984.8,62683.3,7.38,6612.0


## 실행

embedding은 grid의 모든 점에서 동일하므로 **한 번만 계산해 재사용**합니다. penalty 값은 계수의 크기만 바꿀 뿐 어떤 변수쌍이 연결되는지는 바꾸지 않기 때문입니다. 이렇게 해야 관측된 차이가 embedding 운이 아니라 penalty 균형 때문임을 보장할 수 있습니다.

오프라인 모드에서 Pegasus embedding 탐색은 큰 instance에서 수 분이 걸릴 수 있습니다. 빠르게 확인만 하려면 `check_embedding=False`로 두십시오.

In [5]:
from src.persistence import save_table

records = []
for name, formulation in TARGETS:
    instance = CFLPInstance.load(DATA_DIR / f"{name}.json")
    model_name = "SS" if formulation == "SS" else "MS"
    reference = float(
        gurobi[(gurobi["instance"] == name)
               & (gurobi["gurobi_model"] == model_name)]["objective"].iloc[0]
    )
    print(f"--- {name} {formulation} ---")
    points = PB.run_ratio_sweep(
        instance=instance,
        formulation=formulation,
        reference_objective=reference,
        config=config,
        capacity_multiplier=CAPACITY_MULTIPLIER,
        ratios=RATIOS,
        qa_dryrun=QA_DRYRUN,
        check_embedding=True,
    )
    for point in points:
        print(
            f"  ratio={point.ratio:8.0f}  "
            f"feasible={point.feasible_fraction * 100:5.1f}%  "
            f"gap={point.true_gap_percent:7.1f}%  "
            f"range={point.qubo_range:.3e}"
        )
    records.extend(points)

results = pd.DataFrame([point.to_record() for point in records])

--- 4x4 SS ---
  ratio=       1  feasible=  0.4%  gap=    0.4%  range=5.965e+07
  ratio=      10  feasible=  0.5%  gap=   27.7%  range=5.965e+07
  ratio=      50  feasible=  0.4%  gap=   26.7%  range=5.965e+07
  ratio=     100  feasible=  0.6%  gap=   22.8%  range=5.965e+07
  ratio=     200  feasible=  1.4%  gap=    0.4%  range=5.965e+07
  ratio=     500  feasible=  4.0%  gap=    0.0%  range=5.965e+07
  ratio=    1000  feasible=  6.7%  gap=    0.0%  range=5.965e+07
  ratio=    2000  feasible=  5.7%  gap=    0.0%  range=5.965e+07
  ratio=    5000  feasible=  2.1%  gap=    7.3%  range=8.982e+07
--- 6x6 SS ---
  ratio=       1  feasible=  0.0%  gap=    nan%  range=1.513e+08
  ratio=      10  feasible=  0.4%  gap=    7.4%  range=1.513e+08
  ratio=      50  feasible=  0.1%  gap=   27.5%  range=1.513e+08
  ratio=     100  feasible=  0.0%  gap=    nan%  range=1.513e+08
  ratio=     200  feasible=  0.0%  gap=    nan%  range=1.513e+08
  ratio=     500  feasible=  0.4%  gap=   22.5%  range=1.513

## 결과 저장

오프라인과 QPU 결과를 **같은 파일에 누적**합니다. `QA_DRYRUN`을 바꿔 다시 실행하면 두 모드가 함께 쌓여 notebook 13에서 나란히 비교할 수 있습니다.

In [6]:
OUTPUT = PROCESSED_DIR / "penalty_balance.csv"
if OUTPUT.exists():
    previous = pd.read_csv(OUTPUT)
    key = ["instance", "formulation", "mode", "capacity_multiplier", "ratio"]
    merged = pd.concat([previous, results], ignore_index=True)
    merged = merged.drop_duplicates(subset=key, keep="last")
else:
    merged = results
save_table(merged, PROCESSED_DIR, "penalty_balance.csv")
print(f"저장: {OUTPUT} ({len(merged)} 행)")
print(merged.groupby(["mode", "instance", "formulation"]).size())

저장: C:\Users\User\Desktop\KMJ\Study\Quantum\cflp_formulation\results\processed\penalty_balance.csv (36 행)
mode  instance  formulation
qpu   15x15     SS             9
      4x4       SS             9
      6x6       SS             9
      8x8       SS             9
dtype: int64


## 요약

`ratio = 1`(기존 설정) 대비 가장 좋았던 ratio를 비교합니다. `range`가 함께 표시되므로, 개선이 coefficient range를 늘리지 않고 얻어진 것인지 바로 확인할 수 있습니다.

In [7]:
for line in PB.summarize(merged):
    print(line)

4x4 SS (qpu): feasible 0.4% -> 6.7% (ratio=1000, 17배), range 5.96e+07 -> 5.96e+07
6x6 SS (qpu): feasible 0.0% -> 2.5% (ratio=1000, 0에서 상승), range 1.51e+08 -> 1.51e+08
8x8 SS (qpu): feasible 0.0% -> 0.0% (ratio=1, 0에서 상승), range 7.21e+08 -> 7.21e+08
15x15 SS (qpu): feasible 0.0% -> 0.0% (ratio=1, 0에서 상승), range 1.58e+09 -> 1.58e+09


## embedding 정보

오프라인 모드에서는 Pegasus P16(결함 없는 이상적 그래프)에 대한 결과입니다. 실제 QPU는 결함 큐빗이 있어 물리 큐빗 수가 다소 다를 수 있습니다.

In [8]:
columns = [
    column
    for column in (
        "instance", "formulation", "mode", "qubo_variables",
        "qubo_quadratic_terms", "embedding_status", "physical_qubits",
        "max_chain_length", "embedding_search_time",
    )
    if column in merged.columns
]
merged[columns].drop_duplicates(subset=["instance", "formulation", "mode"])

,instance,formulation,mode,qubo_variables,qubo_quadratic_terms,embedding_status,physical_qubits,max_chain_length,embedding_search_time
0,4x4,SS,qpu,44,246,OK,105,4,0.645412
9,6x6,SS,qpu,79,571,OK,271,7,2.544276
18,8x8,SS,qpu,117,1030,OK,619,11,4.211506
27,15x15,SS,qpu,329,5026,OK,4494,34,129.448075
